In [ ]:
%sql
-- compact the fact tables and co-locate the columns used by Power BI filters
USE CATALOG energy;
OPTIMIZE d2_gold.fact_price_hourly ZORDER BY (date_key, zone_code);
OPTIMIZE d2_gold.fact_border_hourly ZORDER BY (date_key, border_key);
OPTIMIZE d2_gold.fact_route_flow_hourly ZORDER BY (date_key, route_key);
OPTIMIZE d2_gold.fact_region_hourly ZORDER BY (date_key);

In [ ]:
%sql
-- rows per table
SELECT 'bronze_prices' AS table_name, COUNT(*) AS n_rows FROM d2_bronze.bronze_prices
UNION ALL SELECT 'bronze_net_position', COUNT(*) FROM d2_bronze.bronze_net_position
UNION ALL SELECT 'bronze_flows', COUNT(*) FROM d2_bronze.bronze_flows
UNION ALL SELECT 'bronze_scheduled', COUNT(*) FROM d2_bronze.bronze_scheduled
UNION ALL SELECT 'fact_price_hourly', COUNT(*) FROM d2_gold.fact_price_hourly
UNION ALL SELECT 'fact_border_hourly', COUNT(*) FROM d2_gold.fact_border_hourly
UNION ALL SELECT 'fact_route_flow_hourly', COUNT(*) FROM d2_gold.fact_route_flow_hourly
UNION ALL SELECT 'fact_region_hourly', COUNT(*) FROM d2_gold.fact_region_hourly;

In [ ]:
%sql
-- duplicate keys (should be empty)
SELECT 'fact_price_hourly' AS table_name, zone_code AS key_1, CAST(ts_utc AS STRING) AS key_2, COUNT(*) AS n
FROM d2_gold.fact_price_hourly GROUP BY zone_code, ts_utc HAVING COUNT(*) > 1
UNION ALL
SELECT 'fact_border_hourly', border_key, CAST(ts_utc AS STRING), COUNT(*)
FROM d2_gold.fact_border_hourly GROUP BY border_key, ts_utc HAVING COUNT(*) > 1;

In [ ]:
%sql
-- prices: orders of magnitude, negative prices, net positions
SELECT
  zone_code,
  COUNT(*) AS n_hours,
  ROUND(AVG(price_eur_mwh), 1) AS avg_price,
  ROUND(MIN(price_eur_mwh), 1) AS min_price,
  ROUND(MAX(price_eur_mwh), 1) AS max_price,
  SUM(CASE WHEN is_negative_price THEN 1 ELSE 0 END) AS negative_hours,
  ROUND(AVG(net_position_mwh)) AS avg_net_position_mw,
  SUM(CASE WHEN net_position_mwh IS NULL THEN 1 ELSE 0 END) AS hours_without_position
FROM d2_gold.fact_price_hourly
GROUP BY zone_code
ORDER BY zone_code;

In [ ]:
%sql
-- borders: convergence, congestion rent, adverse flows
SELECT
  f.border_key,
  COUNT(*) AS n_hours,
  ROUND(AVG(CAST(is_converged AS INT)), 3) AS share_converged,
  ROUND(AVG(abs_spread_eur_mwh), 2) AS avg_abs_spread,
  ROUND(SUM(congestion_rent_eur) / 1e6, 2) AS rent_meur,
  ROUND(AVG(CAST(is_adverse_flow AS INT)), 3) AS share_adverse_flows,
  SUM(CASE WHEN net_flow_a_to_b_mwh IS NULL THEN 1 ELSE 0 END) AS hours_without_flow
FROM d2_gold.fact_border_hourly f
GROUP BY f.border_key
ORDER BY f.border_key;